# Appendix B.2: learning rate and forgetting

The learning-rate sweep: per model, rate, training language and size, how many of the 3 runs reach 256 steps without losing more than 25% of the known facts, and the share lost at the last step reached. Then the known facts lost after 256 steps at the paper's rate, trained in the second language (mean over the 30 facts): 5-7% with S1, 8-12% with S5, 13-22% with all templates. Section 6 of the numbers: the drop per checkpoint.

**Inputs** (GPU, run once; `results/<experiment>/`):
- `python -m factdepth.run experiments/configs/main.yaml`
- `python -m factdepth.run experiments/configs/lr_sweep.yaml`

The outputs below were computed from the paper's own result files; rerunning the configs gives the same numbers.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "factdepth").exists())
sys.path.insert(0, str(ROOT / "src"))
from factdepth import config as C
from factdepth.analysis import numbers, plots, tables
# the results of the configs (results/<experiment>/, or the folder in FACTDEPTH_RESULTS)
print({k: C.results(k).exists() for k in ['main', 'lr_sweep']})

{'main': True, 'lr_sweep': True}


In [2]:
print(tables.lr_sweep())

model | lr | trained in | S1: reached 256 / drop at last step | S3: reached 256 / drop at last step | S86: reached 256 / drop at last step
gemma-3-12b-it | 2e-05 | ZH | 3/3 / -2% | 3/3 / -5% | 3/3 / 3%
gemma-3-12b-it | 2e-05 | EN | 3/3 / -1% | 3/3 / -2% | 1/3 / 19%
gemma-3-12b-it | 6e-05 | ZH | 3/3 / 3% | 3/3 / 4% | 1/3 / 25%
gemma-3-12b-it | 6e-05 | EN | 3/3 / 4% | 2/3 / 12% | 1/3 / 47%
gemma-3-12b-it | 2e-04 | ZH | 0/3 / 40% | 0/3 / 35% | 0/3 / 46%
gemma-3-12b-it | 2e-04 | EN | 1/3 / 41% | 0/3 / 59% | 0/3 / 84%
gemma-3-12b-it | 2e-03 | ZH | 0/3 / 91% | 0/3 / 99% | 0/3 / 100%
gemma-3-12b-it | 2e-03 | EN | 0/3 / 98% | 0/3 / 100% | 0/3 / 100%
Qwen2.5-7B-Instruct | 2e-05 | ZH | 3/3 / -0% | 3/3 / -0% | 2/3 / 15%
Qwen2.5-7B-Instruct | 2e-05 | EN | 3/3 / 3% | 3/3 / 5% | 0/3 / 51%
Qwen2.5-7B-Instruct | 6e-05 | ZH | 3/3 / 9% | 2/3 / 14% | 1/3 / 33%
Qwen2.5-7B-Instruct | 6e-05 | EN | 1/3 / 17% | 2/3 / 20% | 0/3 / 46%
Qwen2.5-7B-Instruct | 2e-04 | ZH | 1/3 / 40% | 1/3 / 35% | 1/3 / 44%
Qwen2.5-

In [3]:
print(tables.forgetting())

model | S1 | S5 | all
Qwen2.5-7B | 5.8 | 11.5 | 21.5
Gemma-3-12B | 4.6 | 7.5 | 12.6
Gemma-3-27B | 7.1 | 11.8 | 21.8


In [4]:
print(numbers.section(numbers.model_numbers("gemma-3-12b-it"), 6))

## 6. Forgetting: drop of the known facts, in percentage points (before - after)

| trained in | size | steps | median | mean | max | runs > 10 pp | runs > 20 pp | intact (ratio <= 25%) | hijack: known prompts naming the injected city (pattern) |
|---|---|---|---|---|---|---|---|---|---|
| 2nd | 1 | 32 | -0.9 | -1.2 | 7.9 | 0/30 | 0/30 | 30/30 | 0% (18/12666) |
| 2nd | 1 | 64 | -1.1 | -0.7 | 8.0 | 0/30 | 0/30 | 30/30 | 0% (63/12666) |
| 2nd | 1 | 128 | -0.6 | -0.6 | 7.8 | 0/30 | 0/30 | 30/30 | 1% (94/12666) |
| 2nd | 1 | 256 | -0.6 | -0.6 | 8.4 | 0/30 | 0/30 | 30/30 | 1% (94/12666) |
| 2nd | 5 | 32 | -2.3 | -1.9 | 5.6 | 0/30 | 0/30 | 30/30 | 0% (25/12666) |
| 2nd | 5 | 64 | -0.3 | -0.7 | 14.8 | 2/30 | 0/30 | 30/30 | 5% (683/12666) |
| 2nd | 5 | 128 | +0.2 | +0.7 | 20.1 | 3/30 | 1/30 | 29/30 | 7% (880/12666) |
| 2nd | 5 | 256 | -0.3 | +0.9 | 20.5 | 3/30 | 1/30 | 29/30 | 7% (913/12666) |
| 2nd | all | 32 | -2.3 | -2.1 | 3.1 | 0/30 | 0/30 | 30/30 | 0% (31/12666) |
| 2nd | all | 64 | +3.2 